# Cellarity public collection processing decisions

Executable reconstruction record for the single biological dataset `cellarity/public-collection` (GEO GSE305370, GSE305979, and GSE306429; DOI 10.1126/science.adi8577). The ten source H5AD families contain 2,212,441 observations. This notebook is Mac-safe: it validates committed manifests and live receipts without querying LaminDB, GCS, or loading X.

## Accepted decisions and boundaries

- OBS revisions preserve exact source row order and OBS UUID identity; each binding canonical field has value, state, and source evidence. Unsupported applicable values remain `unknown`; genuinely irrelevant fields are `not_applicable`.
- Existing X payload bytes and accepted species-correct VAR payloads are retained. Canonical publication remaps their catalog keys without changing UID/hash, and publishes new OBS revisions under `data/cleaned/<source-stem>/`.
- The three immutable successor Collections replace exactly ten predecessor OBS members while preserving all unrelated members; the dataset-level Collection has exactly ten canonical OBS anchors.
- Legacy H5AD is retained because no matrix or chunking defect was identified. Distinct feature axes are not forced into an invented shared VAR.
- The historical task-owned staging prefix is empty. Durable receipts live under `gs://scperturb/data/cleaned/cellarity_public_collection/_receipts/`; upstream GEO objects remain independently reacquirable by immutable URL and SHA-256.

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
root = next((path for path in (cwd, *cwd.parents) if (path / 'src').is_dir() and (path / 'pyproject.toml').is_file()), None)
if root is None:
    raise RuntimeError('Could not locate the pert-gym repository root.')
sys.path.insert(0, str(root / 'src'))
from pert_gym.processing_decisions import validate_processing_decisions_contract

evidence = root / 'artifacts/schema_audit/real_dataset_curation_20260723/cellarity_public_collection/t_9c09e453'
manifest = json.loads((evidence / 'source_manifest.json').read_text())
receipt_index = json.loads((evidence / 'live_receipt_index.json').read_text())
plan = json.loads((evidence / 'publication_plan_receipt.json').read_text())
mutation = json.loads((evidence / 'authoritative_mutation_receipt.json').read_text())
verify = json.loads((evidence / 'zero_write_verify_receipt.json').read_text())

In [ ]:
def canonical_sha256(receipt):
    unsigned = {key: value for key, value in receipt.items() if key != 'canonical_sha256'}
    payload = json.dumps(unsigned, sort_keys=True, separators=(',', ':'), default=str).encode()
    return hashlib.sha256(payload).hexdigest()

for receipt in (plan, mutation, verify):
    assert receipt['status'] == 'PASS'
    assert receipt['canonical_sha256'] == canonical_sha256(receipt)
    assert len(receipt['members']) == 10
    assert receipt['source_denominator'] == {'biological_datasets': 1, 'logical_families': 10, 'physical_members': 10, 'observations': 2212441}
    assert receipt['gcs_decommission']['GCS_DECOMMISSION_READY'] is True
    assert receipt['gcs_decommission']['objects_remaining'] == 0
assert plan['mode'] == 'plan' and plan['registry_counts']['before'] == plan['registry_counts']['after']
assert mutation['mode'] == 'mutate'
assert mutation['writes']['obs_revisions'] == 10
assert mutation['writes']['collection_writes'] == 3
assert len(mutation['writes']['canonical_key_remaps']) == 20
assert mutation['posthoc_var_verification_binding']['source_receipt_canonical_sha256'] == verify['canonical_sha256']
assert mutation['posthoc_var_verification_binding']['original_mutation_receipt_canonical_sha256'] == receipt_index['receipts']['authoritative_mutation']['remote_canonical_sha256']
assert mutation['registry_counts'] == {'before': {'artifacts': 28590, 'collections': 52}, 'after': {'artifacts': 28600, 'collections': 55}}
assert verify['mode'] == 'verify' and verify['replay_noop'] is True
assert verify['registry_counts']['before'] == verify['registry_counts']['after'] == {'artifacts': 28600, 'collections': 55}
assert verify['writes']['obs_revisions'] == verify['writes']['collection_writes'] == 0
assert all(member['already_curated'] for member in verify['members'])
assert all(member['obs']['key'].startswith('data/cleaned/') for member in verify['members'])
assert all(member['x']['key'].startswith('data/cleaned/') for member in verify['members'])
assert all(member['var']['key'].startswith('data/cleaned/') for member in verify['members'])

In [ ]:
sources = manifest['target_source_objects']
assert len(sources) == 10
assert sum(item['n_obs'] for item in sources) == 2212441
assert {item['accession'] for item in sources} == {'GSE305370', 'GSE305979', 'GSE306429'}
assert all(item['url'].startswith('https://ftp.ncbi.nlm.nih.gov/geo/') for item in sources)
assert all(len(item['sha256']) == 64 and item['size'] > 0 for item in sources)
assert all(member['source_join']['exact_index_order_match'] for member in verify['members'])
for receipt in (mutation, verify):
    assert all(member['var_verification']['VAR_ENSEMBL_SPECIES_COMPLETED'] for member in receipt['members'])
    assert all(len(member['var_verification']['ordered_var_axis_sha256']) == 64 for member in receipt['members'])
    assert {member['var_verification']['axis_match_mode'] for member in receipt['members']} == {'byte_exact', 'exact_ordered_case_normalization_bijection'}
for receipt_name, receipt in [('publication_plan', plan), ('authoritative_mutation', mutation), ('zero_write_verify', verify)]:
    indexed = receipt_index['receipts'][receipt_name]
    assert indexed['canonical_sha256'] == receipt['canonical_sha256']
    assert indexed['remote']['uri'].startswith('gs://scperturb/data/cleaned/cellarity_public_collection/_receipts/')

In [ ]:
collection_keys = sorted(verify['published_collections'])
contract = {
    'identity': {
        'dataset_id': 'cellarity/public-collection',
        'source': 'GEO GSE305370, GSE305979, GSE306429; DOI 10.1126/science.adi8577',
        'version': 'GEO source objects frozen by per-file SHA-256',
        'license': 'GEO public access; source publication terms apply',
        'checksums': {item['filename']: f"sha256:{item['sha256']}" for item in sources},
    },
    'delta_vs_main': {
        'branch': 'jkobject',
        'artifact_count': 10,
        'logical_dataset_count': 1,
        'collection_count': 3,
        'added_artifacts': [{'key': member['obs']['key'], 'uid': member['obs']['uid']} for member in mutation['members']],
        'added_collections': collection_keys,
    },
    'biological_context': {'unit': 'cell or source-defined aggregate observation', 'modality': 'single-cell and pseudobulk RNA response with CITE-seq/multiome source families'},
    'source_payload': {'included': [item['filename'] for item in sources], 'excluded': []},
    'processing_decisions': {
        'inclusion': 'All ten GEO H5AD families in the frozen Cellarity public-collection crosswalk.',
        'exclusion': 'No target family excluded; unsupported fields remain explicitly unknown.',
        'conversion': 'OBS-only append-only Parquet revisions; accepted X/VAR bytes retained.',
        'transformations': 'Source-backed metadata normalization with exact row joins and explicit field state/source columns.',
        'quality_control': 'No source rows filtered; row count, order, and obs_uuid are preserved.',
        'obs_schema': f"Binding {verify['obs_contract']['contract_id']} with {verify['obs_contract']['canonical_field_count']} canonical fields.",
        'perturbation_mapping': 'Only source H5AD treatment, compound, dose, and target evidence is materialized.',
        'control_mapping': 'Vehicle/no-treatment controls are source-derived; observational families are not mislabeled as controls.',
        'organism_and_gene_normalization': 'Homo sapiens; every biological VAR feature passes exact human Ensembl/species verification.',
        'x_semantics': 'Preserved per-family source semantics; unknown transforms remain explicitly unknown.',
        'chunk_size_policy': 'No matrix rewrite or rechunking; ten source-defined physical members retained.',
        'zarr_or_h5ad': 'Legacy H5AD retained because no structural or payload defect required conversion.',
        'shared_var_identity': 'No forced shared VAR across distinct axes. Seven axes are byte-exact; the three GSE306429 axes have an immutable-source-bound, collision-free ordered case-normalization bijection (170 symbols), with exact stable human Ensembl IDs.',
        'auxiliary_modalities': 'CITE-seq protein and multiome ATAC context retained in source identity; canonical X links are unchanged.',
    },
    'rejected_alternatives': [
        {'alternative': 'rewrite X or force one shared VAR', 'reason_rejected': 'No named parity defect and source families do not share one proven identical axis.'},
        {'alternative': 'invent missing metadata from publication constants', 'reason_rejected': 'No defensible row-level join.'},
    ],
    'lineage': {
        'raw_and_staged_uris': [item['url'] for item in sources],
        'script': 'artifacts/schema_audit/real_dataset_curation_20260723/cellarity_public_collection/t_9c09e453/curate_obs.py',
        'commit': 'live receipts bind helper_sha256; PR #120 records immutable Git lineage',
        'card_id': 't_9c09e453 continued by t_60d97299',
        'legacy_to_logical_map': {member['identity']['prefix']: member['obs']['key'] for member in verify['members']},
        'branch_policy': 'Lamin branch jkobject only; no main writes.',
    },
    'validation': {
        'readback': f"PASS mutation {mutation['canonical_sha256']}; zero-write verify {verify['canonical_sha256']}",
        'denominator': '1 biological dataset; 10 logical families; 10 physical members; 2,212,441 observations',
    },
    'collection_membership': {'collections': collection_keys, 'model_ready_query': 'Dataset Collection has exactly ten canonical OBS anchors; follow OBS -> X -> VAR links.'},
    'limitations_and_rollback': {
        'limitations': ['Some applicable metadata remains honestly unknown where sources provide no defensible row value.'],
        'rollback': mutation['rollback_identity'],
        'retention': 'Immutable upstream GEO H5ADs, predecessor Lamin revisions, and durable cleaned receipts.',
    },
    'temporary_gcs_dependencies': [],
    'reconstruction': {
        'reproducibility_claimed': True,
        'immutable_upstream_sources': [{'uri': item['url'], 'sha256': item['sha256']} for item in sources],
        'retained_lamin_raw_artifact': None,
        'safe_to_remove_gcs': True,
        'procedure': 'Fetch each generation-independent GEO URL, verify SHA-256, and rerun the source inspection and curation helper on the bounded EU worker.',
    },
    'runtime': {'live_lamin_query_enabled': False, 'allowed_live_lamin_hosts': ['pert-gym-worker-eu']},
}
errors = validate_processing_decisions_contract(contract)
assert not errors, '\n'.join(errors)

## Residual gates

The committed receipts establish durable `jkobject` publication and a zero-write replay. Independent review must still validate this notebook, receipt identities, semantic field decisions, and exact PR head before merge. Merge status is deliberately not represented as scientific evidence.